In [8]:
# ================================
# UPLOAD + CLEANING + CHURN PROXY
# ================================

import os
import pandas as pd

# Se não tiver o CSV no runtime, pedir upload
if not os.path.exists("revolut_reviews.csv"):
    from google.colab import files
    print("⬆️ Faça upload do arquivo revolut_reviews.csv")
    files.upload()

# Carregar CSV
df_raw = pd.read_csv("revolut_reviews.csv")

# Padronizar colunas
df_raw.columns = [c.strip().lower() for c in df_raw.columns]

# Detectar colunas automaticamente
rating_col = next((c for c in ["rating","stars","score"] if c in df_raw.columns), None)
text_col = next((c for c in ["review_text","review","text","content","comment","body"] if c in df_raw.columns), None)

if rating_col is None or text_col is None:
    raise Exception(f"Colunas não detectadas. Colunas disponíveis: {df_raw.columns.tolist()}")

# Cleaning
df_raw["rating"] = pd.to_numeric(df_raw[rating_col], errors="coerce")
df_raw["review_text"] = df_raw[text_col].astype(str).str.strip().str.lower()
df_raw = df_raw.dropna(subset=["rating","review_text"])
df_raw["rating"] = df_raw["rating"].astype(int)

df_raw["review_len"] = df_raw["review_text"].str.len()
df_raw["churn_risk"] = (df_raw["rating"] <= 2).astype(int)

df = df_raw[["rating","review_text","review_len","churn_risk"]].copy()

print("✅ Dataset pronto")
df.head(), df.shape

✅ Dataset pronto


(   rating                                        review_text  review_len  \
 0       5  good bank but kept trying to change my current...         259   
 1       5                                great, easy to use.          19   
 2       5                                               good           4   
 3       5  i think it's a really handy app to have, i was...         179   
 4       1  i was doing an order before midnight to avoid ...         227   
 
    churn_risk  
 0           0  
 1           0  
 2           0  
 3           0  
 4           1  ,
 (1200, 4))

In [9]:
# ================================
# SENTIMENT ANALYSIS
# ================================

from textblob import TextBlob

df["sentiment_polarity"] = df["review_text"].apply(
    lambda x: TextBlob(x).sentiment.polarity
)

df.head()

,rating,review_text,review_len,churn_risk,sentiment_polarity
0,5,good bank but kept trying to change my current...,259,0,0.366667
1,5,"great, easy to use.",19,0,0.616667
2,5,good,4,0,0.700000
3,5,"i think it's a really handy app to have, i was...",179,0,0.300000
4,1,i was doing an order before midnight to avoid ...,227,1,0.225000


In [10]:
# ==========================================
# INSIGHT: 1★ nem sempre é extremamente negativa
# ==========================================

one_star = df[df["rating"] == 1].copy()

one_star["sentiment_bucket"] = pd.cut(
    one_star["sentiment_polarity"],
    bins=[-1, -0.6, -0.2, 0.2, 1],
    labels=["very_negative", "negative", "neutralish", "positive"]
)

one_star["sentiment_bucket"].value_counts()

,count
sentiment_bucket,
neutralish,60
negative,16
positive,9
very_negative,3


In [11]:
# ================================
# MODELO DE SATISFAÇÃO
# ================================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

X = df[["sentiment_polarity", "review_len"]]
y = df["rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R2:", r2_score(y_test, pred))

MAE: 0.5635910749127238
R2: 0.28872447215803
